In [1]:
from collections import deque
from dash.dependencies import Output, Input
from dash import dcc
from dash import html
from PyQt5 import QtWidgets
import threading as th
import dash
import plotly
import webbrowser


import pandas as pd
import sys
import re
import serial
import serial.tools.list_ports as serialport
import plotly.figure_factory as ff
import plotly.graph_objs as go
import plotly.express as px
import subprocess

In [2]:
# sample data:
data = [{'value': 10, 'onset': 1.0, 'offset': 2.0, 'duration': 1.0, 'occurence': 1},
        {'value': 20, 'onset': 4.0, 'offset': 4.1, 'duration': 0.1, 'occurence': 1},
        {'value': 30, 'onset': 6.0, 'offset': 9.0, 'duration': 3.0, 'occurence': 1},
        {'value': 40, 'onset': 9.0, 'offset': 11.0, 'duration': 2.0, 'occurence': 1},
        {'value': 10, 'onset': 12.0, 'offset': 14.0, 'duration': 2.0, 'occurence': 2}
]

data_frame = pd.DataFrame(data= data, columns = ['value', 'onset', 'offset', 'duration', 'occurence'])

summary = data_frame
summary = summary[['value', 'occurence']]
summary = summary.drop_duplicates(subset = ['value'], keep = 'last')

print("The data frame:\n", data_frame, "\n\n\n The summaries table:\n", summary)

The data frame:
    value  onset  offset  duration  occurence
0     10    1.0     2.0       1.0          1
1     20    4.0     4.1       0.1          1
2     30    6.0     9.0       3.0          1
3     40    9.0    11.0       2.0          1
4     10   12.0    14.0       2.0          2 


 The summaries table:
    value  occurence
1     20          1
2     30          1
3     40          1
4     10          2


In [11]:
# Initialize a figure:
table = ff.create_table(data_frame)

temp = data_frame

# Graph data:
values = temp[['value']]
times = temp[['onset']]    # TODO: gotta figure this out


# Plot the figure:
table.show()

# Scatter plot:
plot = px.histogram(data_frame = data_frame,
                    x = 'onset',
                    y = 'value',
                    title = 'Markers plotted as digital signals',
                    labels = {'onset': 'Time (micro seconds)', 'value': 'Marker Value'},
                    color = 'value',
                    marginal = 'rug')


plot.show()

In [5]:
X = deque([], maxlen = 100)
Y = deque([], maxlen = 100)

# Initialize and setup the dash app:
def run_dash_app():
    global app
    app = dash.Dash(__name__)

    app.layout = html.Div(
        [
            dcc.Graph(id = 'Live marker plot',
                    animate = True),
            
            dcc.Interval(id = 'update-graph',
                            interval = 1000,
                            n_intervals = 0),
        ]
    )

    app.run_server(debug = False)

# Callback decorator:
@app.callback(
    Output('Live marker plot', 'figure'),
    [Input('update-graph', 'n_intervals')]
)

# Update graph:
def update_graph(n, data):

    global X, Y 
    # Get the data:
    X.append(data['onset'])
    Y.append(data['value'])

    data = go.Scatter(x = list(X),
                      y = list(Y),
                      name = 'Marker Value',
                      mode = 'lines+markers')

    return {'data': [data],
            'layout': go.Layout(xaxis = {'title': 'Time (micro seconds)'},
                                yaxis = {'title': 'Marker Value'})}



class MainWindow(QtWidgets.QMainWindow):
    pass

if __name__ == '__main__':
    layout = {'title': 'Markers plotted as digital signals'}

    th.Thread(target=run_dash_app(), args=(data, layout), daemon=True).start()
    app = QtWidgets.QApplication(sys.argv)
    mainWin = MainWindow()
    mainWin.show()
    sys.exit(app.exec_())

NameError: name 'app' is not defined

In [ ]:
import datetime

import dash
from dash import dcc, html
import plotly
from dash.dependencies import Input, Output

# pip install pyorbital
from pyorbital.orbital import Orbital
satellite = Orbital('TERRA')

external_stylesheets = ['https://codepen.io/chriddyp/pen/bWLwgP.css']

app = dash.Dash(__name__, external_stylesheets=external_stylesheets)
app.layout = html.Div(
    html.Div([
        html.H4('TERRA Satellite Live Feed'),
        html.Div(id='live-update-text'),
        dcc.Graph(id='live-update-graph'),
        dcc.Interval(
            id='interval-component',
            interval=1*1000, # in milliseconds
            n_intervals=0
        )
    ])
)


@app.callback(Output('live-update-text', 'children'),
              Input('interval-component', 'n_intervals'))
def update_metrics(n):
    lon, lat, alt = satellite.get_lonlatalt(datetime.datetime.now())
    style = {'padding': '5px', 'fontSize': '16px'}
    return [
        html.Span('Longitude: {0:.2f}'.format(lon), style=style),
        html.Span('Latitude: {0:.2f}'.format(lat), style=style),
        html.Span('Altitude: {0:0.2f}'.format(alt), style=style)
    ]


# Multiple components can update everytime interval gets fired.
@app.callback(Output('live-update-graph', 'figure'),
              Input('interval-component', 'n_intervals'))
def update_graph_live(n):
    satellite = Orbital('TERRA')
    data = {
        'time': [],
        'Latitude': [],
        'Longitude': [],
        'Altitude': []
    }

    # Collect some data
    for i in range(180):
        time = datetime.datetime.now() - datetime.timedelta(seconds=i*20)
        lon, lat, alt = satellite.get_lonlatalt(
            time
        )
        data['Longitude'].append(lon)
        data['Latitude'].append(lat)
        data['Altitude'].append(alt)
        data['time'].append(time)

    # Create the graph with subplots
    fig = plotly.tools.make_subplots(rows=2, cols=1, vertical_spacing=0.2)
    fig['layout']['margin'] = {
        'l': 30, 'r': 10, 'b': 30, 't': 10
    }
    fig['layout']['legend'] = {'x': 0, 'y': 1, 'xanchor': 'left'}

    fig.append_trace({
        'x': data['time'],
        'y': data['Altitude'],
        'name': 'Altitude',
        'mode': 'lines+markers',
        'type': 'scatter'
    }, 1, 1)
    fig.append_trace({
        'x': data['Longitude'],
        'y': data['Latitude'],
        'text': data['time'],
        'name': 'Longitude vs Latitude',
        'mode': 'lines+markers',
        'type': 'scatter'
    }, 2, 1)

    return fig


if __name__ == '__main__':
    app.run_server(debug=True)

In [ ]:
def test_serial(dev_name = ''):
    """ returns open serial ports (COM_portnumber) identified by string matching with desired_ports[]

        :raises EnvironmentError:
            On unsupported or unknown platforms
        :returns:
            A list of the serial ports available on the system
    """
    # check windows OS:
    if sys.platform.startswith('win'):
        com_ports = serialport.comports()
    else:
        raise EnvironmentError('Unsupported platform')

    device_name = []
    property = []
    device_number = 0
    counter = []   # for detecting chosen device connected or not
    desired_ports = ['USB Serial Device', 'Ardruino', 'Lombardo']   # specify the desired device descriptions here to choose the desired device

    for port in com_ports:
        try:
            if (desired_ports[0] or desired_ports[1] or desired_ports[2]) in port.description:
                device_number += 1
                # Initialize the COM port:
                s = serial.Serial(port.device)
                if s.isOpen():
                    if dev_name != '':
                        if s._port_handle != dev_name:
                            counter.append(1)
                    print(s.portstr)
                    device_name.append(s.port)
                    property.append(port.hwid)    # define any desirable property instead of hwid
                    s.close()
        except (OSError, serial.SerialException):
            pass
    
    print("Found", device_number, "UsbParMar devices.")
    if not counter:
        pass
    else:
        raise Exception("Connected device(s) did not match the requested device")

    print(property)

    return device_name


print("Available COM ports: ", test_serial())

In [ ]:
AllocatedResources = subprocess.Popen("wmic path Win32_PNPAllocatedResource Get", \
                    shell=True, stdout=subprocess.PIPE).stdout.read().splitlines()

PNPDeviceIDstrings = subprocess.Popen("wmic path Win32_ParallelPort Get PNPDeviceID /VALUE", \
                               shell=True, stdout=subprocess.PIPE).stdout.read()

PNPDeviceIDs = []
for line in PNPDeviceIDstrings.splitlines():
    if line != b'':
        PNPDeviceID = str(line.strip())
        PNPDeviceID = PNPDeviceID.replace('\'','')
        PNPDeviceID = PNPDeviceID.replace('\\\\','\\')
        PNPDeviceID = PNPDeviceID.replace('&amp;','&')
        PNPDeviceID = PNPDeviceID.split('=')[1]
        PNPDeviceID = PNPDeviceID.encode()
        PNPDeviceIDs.append(PNPDeviceID)

# list available LPT port addresses
AddressesFound = []
for r in AllocatedResources:
    for PNPDeviceID in PNPDeviceIDs:
        if r.find(PNPDeviceID) != -1:
            StartingAddresses = re.findall('StartingAddress="[\d]+"', r.decode())
            HexStartingAddress = hex(int(re.findall('[\d]+', StartingAddresses[0])[0]))
            AddressesFound.append(HexStartingAddress[2:].upper())

NumAddressesFound = len(AddressesFound)
print(NumAddressesFound)
print(AddressesFound)

# check port address
if len(sys.argv) == 1: # no port specified to check
    print("False")
    print("Warning: Missing argument. No port specified to check.")
else:
    if sys.argv[1] in AddressesFound:
        print("True") # Port is a correct port
    else:
        print("False")

print("Found " + str(int(NumAddressesFound/2)) + " LPT address(es): ")
for i in range(0,NumAddressesFound,2):
    print(AddressesFound[i]) 